In [0]:
%run ./00_config

In [0]:
from pyspark.sql import functions as F

#### Passo 1.1 — Ler os arquivos originai

In [0]:
fraud_train_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(f"{RAW_VOLUME}/{FILES['fraud_train']}")
)

fraud_test_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(f"{RAW_VOLUME}/{FILES['fraud_test']}")
)

In [0]:
display(fraud_train_raw.limit(10))
display(fraud_test_raw.limit(10))

#### ## Passo 1.2 — Conferir campos essenciais, linhas e colunas


In [0]:
campos_essenciais = {
    "trans_num",
    "cc_num",
    "trans_date_trans_time",
    "amt",
    "is_fraud",
    "merchant",
    "category"
}

faltantes_train = campos_essenciais - set(fraud_train_raw.columns)
faltantes_test = campos_essenciais - set(fraud_test_raw.columns)

print("Faltantes train:", faltantes_train)
print("Faltantes test:", faltantes_test)

print("Train - linhas:", fraud_train_raw.count())
print("Train - colunas:", len(fraud_train_raw.columns))

print("Test - linhas:", fraud_test_raw.count())
print("Test - colunas:", len(fraud_test_raw.columns))

#### Passo 1.3 — Entender a granularidade


In [0]:
linhas_train = fraud_train_raw.count()
tx_distintas_train = fraud_train_raw.select("trans_num").distinct().count()

print("Train - linhas:", linhas_train)
print("Train - trans_num distintos:", tx_distintas_train)

In [0]:
linhas_test = fraud_test_raw.count()
tx_distintas_test = fraud_test_raw.select("trans_num").distinct().count()

print("Test - linhas:", linhas_test)
print("Test - trans_num distintos:", tx_distintas_test)

In [0]:
cartoes_distintos_train = fraud_train_raw.select("cc_num").distinct().count()
cartoes_distintos_test = fraud_test_raw.select("cc_num").distinct().count()

print("Train - cartões distintos:", cartoes_distintos_train)
print("Test - cartões distintos:", cartoes_distintos_test)

In [0]:
(
    fraud_train_raw
    .groupBy("cc_num")
    .count()
    .orderBy(F.desc("count"))
    .show(10)
)

#### Passo 1.5 — Gravar a camada Bronze em Delta


In [0]:
(
    fraud_train_raw.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLE_BRONZE_TRAIN)
)

(
    fraud_test_raw.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLE_BRONZE_TEST)
)

In [0]:
print("Tabela Bronze train:", TABLE_BRONZE_TRAIN)
print("Tabela Bronze test:", TABLE_BRONZE_TEST)

print("Bronze train - linhas:", spark.table(TABLE_BRONZE_TRAIN).count())
print("Bronze test - linhas:", spark.table(TABLE_BRONZE_TEST).count())

In [0]:
print("Tabela Bronze train:", TABLE_BRONZE_TRAIN)